# Loading the CSV file

In [11]:
import pandas as pd

df = pd.read_csv("medical_notes_dataset.csv")

df['combined_data'] = df[['note', 'full_note', 'conversation', 'summary']].fillna('').agg(' '.join, axis=1)

print(df['combined_data'])

0      A 35-year-old gravida 6, para 5 mother who is ...
1      A 57-year-old male who had no family history o...
2      The present case report is about a 72-year-old...
3      Case 1: Ms. K, a 70-year-old woman who immigra...
4      An 11-year-old male in permanent dentition was...
                             ...                        
495    The patient was a 13-year-old girl who suffere...
496    A 70-year-old male patient presented to our em...
497    We report the case of a 40-year old male patie...
498    A 27-year-old man underwent surgical intervent...
499    A 44-year-old man presented to our hospital co...
Name: combined_data, Length: 500, dtype: object


# Chunking the Text

In [12]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)

df['chunks'] = df['combined_data'].apply(lambda x: text_splitter.split_text(x))

# Embedding

In [13]:
from langchain.embeddings import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embeddings = []
metadata = []

for idx, row in df.iterrows():
    patient_id = row['patient_id']
    chunks = row['chunks']
    
    for chunk in chunks:
        embeddings.append(embeddings_model.embed_query(chunk))
        metadata.append({'patient_id': patient_id})

# Setting up Qdrant for VectorDB

In [18]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import VectorParams

# Connect to Qdrant
qdrant_client = QdrantClient(url="http://localhost:6333")

collection_name = "medical_notes"

# Check if the collection exists
if qdrant_client.collection_exists(collection_name):
    # If it exists, delete it
    qdrant_client.delete_collection(collection_name)

# Create a new collection
qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=len(embeddings[0]), distance="Cosine")
)

qdrant_client.upload_collection(
    collection_name=collection_name,
    vectors=embeddings,
    payload=metadata,
    ids=None  
)

#Testing for semantic searching
query = "Symptoms of asthma"
query_embedding = embeddings_model.embed_query(query)

# Search in Qdrant
results = qdrant_client.search(
    collection_name="medical_notes",
    query_vector=query_embedding,
    limit=5
)

# Display results
for result in results:
    print(f"Patient ID: {result.payload['patient_id']}")

Patient ID: p032
Patient ID: p281
Patient ID: p386
Patient ID: p419
Patient ID: p281
